# Things to research

*   How was the data split up, and we should problaly do around 100mb per shard to mimic nanochat
*   Why did the tokenizer eval take 15+ mins?
*   redo data processing / download, maybe try to skip the google drive step like karpathy and download from hugging face
*   how can i save the model before going to training
* is there a way we can remove the google drive variable and go directly from hugging face?
* try to minimize this script and make it easy to swap out for another dataset

# Run the model
* after researching, run the model from the bottom to the top
* hope for ~54 mfu and around 2 hours of processing time
* make sure to save a copy of the pre supervised learning model




In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Edit these before each run. Everything downstream reads from these variables.
MODEL_NAME  = "model_gutenberg_v1"
MODEL_DEPTH = 12
MODEL_TAG   = f"d{MODEL_DEPTH}"

DRIVE_ROOT  = "/content/drive/MyDrive/Think.Genesis"
LOCAL       = "/content/nanochat_cache"

import os
os.environ["MODEL_NAME"]        = MODEL_NAME
os.environ["MODEL_TAG"]         = MODEL_TAG
os.environ["MODEL_DEPTH"]       = str(MODEL_DEPTH)
os.environ["NANOCHAT_BASE_DIR"] = LOCAL
os.environ["DRIVE_ROOT"]        = DRIVE_ROOT
os.environ["LOCAL"]             = LOCAL

print(f"Model : {MODEL_NAME}  depth={MODEL_DEPTH}  tag={MODEL_TAG}")
print(f"Drive : {DRIVE_ROOT}")
print(f"Local : {LOCAL}")


Model : model_gutenberg_v1  depth=12  tag=d12
Drive : /content/drive/MyDrive/Think.Genesis
Local : /content/nanochat_cache


In [ ]:
# ── Clone nanochat + install dependencies ─────────────────────────────────────
# Clones to Drive for persistence; symlinks to /content/nanochat for fast I/O.
# pip install is idempotent.
import os
from google.colab import drive

DRIVE_ROOT = os.environ["DRIVE_ROOT"]

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

NANOCHAT_DRIVE = f"{DRIVE_ROOT}/nanochat"
NANOCHAT_LOCAL = "/content/nanochat"

if not os.path.isdir(NANOCHAT_DRIVE):
    os.makedirs(os.path.dirname(NANOCHAT_DRIVE), exist_ok=True)
    !git clone https://github.com/karpathy/nanochat {NANOCHAT_DRIVE}

if not os.path.exists(NANOCHAT_LOCAL):
    os.symlink(NANOCHAT_DRIVE, NANOCHAT_LOCAL)

%cd /content/nanochat
!pip install -q rustbpe tiktoken tokenizers datasets wandb fastapi uvicorn psutil kernels


Mounted at /content/drive
/content/drive/MyDrive/Think.Genesis/nanochat
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.7 MB/s eta 0:00:00


In [ ]:
# ── Mount Drive + symlink checkpoints/tokenizer ───────────────────────────────
# Checkpoint dirs and tokenizer live on Drive, symlinked into LOCAL so nanochat
# finds them at its hardcoded paths. Re-run safe.
import os
from google.colab import drive

DRIVE_ROOT  = os.environ["DRIVE_ROOT"]
LOCAL       = os.environ["LOCAL"]
MODEL_NAME  = os.environ["MODEL_NAME"]
MODEL_DRIVE = f"{DRIVE_ROOT}/models/{MODEL_NAME}"

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

os.makedirs(LOCAL, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/tokenizer",              exist_ok=True)
os.makedirs(f"{MODEL_DRIVE}/base_checkpoints",      exist_ok=True)
os.makedirs(f"{MODEL_DRIVE}/chatsft_checkpoints",   exist_ok=True)
os.makedirs(f"{MODEL_DRIVE}/chatrl_checkpoints",    exist_ok=True)

def ensure_symlink(link_path, target):
    if os.path.islink(link_path):
        if os.readlink(link_path) == target:
            return
        os.unlink(link_path)
    elif os.path.exists(link_path):
        raise RuntimeError(f"{link_path} exists and is not a symlink; remove manually")
    os.symlink(target, link_path)

ensure_symlink(f"{LOCAL}/tokenizer",           f"{DRIVE_ROOT}/tokenizer")
ensure_symlink(f"{LOCAL}/base_checkpoints",    f"{MODEL_DRIVE}/base_checkpoints")
ensure_symlink(f"{LOCAL}/chatsft_checkpoints", f"{MODEL_DRIVE}/chatsft_checkpoints")
ensure_symlink(f"{LOCAL}/chatrl_checkpoints",  f"{MODEL_DRIVE}/chatrl_checkpoints")

print("Base dir:", LOCAL)
for sub in ["tokenizer", "base_checkpoints", "chatsft_checkpoints", "chatrl_checkpoints"]:
    print(f"  {sub:25s} -> {os.readlink(f'{LOCAL}/{sub}')}")


Drive already mounted.
Base dir: /content/nanochat_cache
  tokenizer                 -> /content/drive/MyDrive/Think.Genesis/tokenizer
  base_checkpoints          -> /content/drive/MyDrive/Think.Genesis/models/model_gutenberg_v1/base_checkpoints
  chatsft_checkpoints       -> /content/drive/MyDrive/Think.Genesis/models/model_gutenberg_v1/chatsft_checkpoints
  chatrl_checkpoints        -> /content/drive/MyDrive/Think.Genesis/models/model_gutenberg_v1/chatrl_checkpoints


In [ ]:
# ── GPU sanity check ──────────────────────────────────────────────────────────
# Verify A100, bf16 support, and all imports resolve correctly.
import torch, rustbpe, tiktoken
print(f"torch          : {torch.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")
print(f"gpu            : {torch.cuda.get_device_name(0)}")
print(f"capability     : sm{''.join(map(str, torch.cuda.get_device_capability(0)))}")
print(f"bf16 supported : {torch.cuda.is_bf16_supported()}")


torch          : 2.11.0+cu128
cuda available : True
gpu            : NVIDIA A100-SXM4-40GB
capability     : sm80
bf16 supported : True


In [ ]:
# ── Download Project Gutenberg dataset ────────────────────────────────────────
# Downloads sedthh/gutenberg_english, splits into 8 equal parquet shards on
# Drive, then copies to local SSD for fast streaming. Re-run safe.
import os, json, time, shutil
from datasets import load_dataset
import pyarrow as pa
import pyarrow.parquet as pq

DRIVE_ROOT = os.environ["DRIVE_ROOT"]
LOCAL      = os.environ["LOCAL"]
NUM_SHARDS = 8
DRIVE_DATA = f"{DRIVE_ROOT}/datasets/Gutenberg"
LOCAL_DATA = f"{LOCAL}/base_data_climbmix"   # nanochat hardcodes this name

os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(LOCAL_DATA,  exist_ok=True)

progress_path = f"{DRIVE_DATA}/_progress.json"
already_done = False
if os.path.exists(progress_path):
    with open(progress_path) as f:
        p = json.load(f)
    if p.get("complete"):
        print(f"Drive: already complete ({p['total_books']:,} books). Syncing to local SSD...")
        already_done = True

if not already_done:
    print("Loading Gutenberg from HuggingFace...")
    ds = load_dataset("sedthh/gutenberg_english", split="train")
    total = len(ds)
    print(f"Books: {total:,}  ->  {NUM_SHARDS} shards of ~{total//NUM_SHARDS:,} each")

    rows_per_shard = total // NUM_SHARDS
    t0 = time.time()
    for idx in range(NUM_SHARDS):
        shard_path = f"{DRIVE_DATA}/shard_{idx:05d}.parquet"
        if os.path.exists(shard_path):
            print(f"  shard_{idx:05d}.parquet already on Drive, skipping.")
            continue
        start = idx * rows_per_shard
        end   = total if idx == NUM_SHARDS - 1 else start + rows_per_shard
        texts = [
            (row.get("TEXT") or "").strip()
            for row in ds.select(range(start, end))
            if (row.get("TEXT") or "").strip()
        ]
        tmp = shard_path + ".tmp"
        pq.write_table(pa.table({"text": texts}), tmp, compression="snappy")
        os.replace(tmp, shard_path)
        print(f"  wrote shard_{idx:05d}.parquet  ({len(texts):,} books)  elapsed={time.time()-t0:.0f}s")

    with open(progress_path, "w") as f:
        json.dump({"complete": True, "total_books": total, "num_shards": NUM_SHARDS}, f)
    print(f"Drive download complete in {time.time()-t0:.0f}s")

# Sync shards from Drive to local SSD
print("Syncing shards Drive -> local SSD...")
for fname in sorted(os.listdir(DRIVE_DATA)):
    if not fname.endswith(".parquet"):
        continue
    dst = f"{LOCAL_DATA}/{fname}"
    if not os.path.exists(dst):
        shutil.copy2(f"{DRIVE_DATA}/{fname}", dst)
        print(f"  copied {fname}")
    else:
        print(f"  {fname} already local.")

n_local = len([f for f in os.listdir(LOCAL_DATA) if f.endswith(".parquet")])
print(f"Local shards ready: {n_local}")


Drive: already complete (48,284 books). Syncing to local SSD...
Syncing shards Drive -> local SSD...
  copied shard_00000.parquet
  copied shard_00001.parquet
  copied shard_00002.parquet
  copied shard_00003.parquet
  copied shard_00004.parquet
  copied shard_00005.parquet
  copied shard_00006.parquet
  copied shard_00007.parquet
Local shards ready: 8


In [ ]:
# ── Train tokenizer ───────────────────────────────────────────────────────────
# Builds a BPE tokenizer from the pretraining data (~3-5 min smoke test,
# ~20-40 min full). Saves to LOCAL/tokenizer/ which is symlinked to Drive.
%cd /content/nanochat
!python -m scripts.tok_train --max-chars=200000000
!python -m scripts.tok_eval


/content/drive/MyDrive/Think.Genesis/nanochat
max_chars: 200,000,000
doc_cap: 10,000
vocab_size: 32,768
2026-05-27 15:26:41,981 - rustbpe - INFO - Processing sequences from iterator (buffer_size: 8192)
2026-05-27 15:27:38,278 - rustbpe - INFO - Processed 20377 sequences total, 576556 unique
2026-05-27 15:27:38,315 - rustbpe - INFO - Starting BPE training: 32503 merges to compute
2026-05-27 15:27:38,316 - rustbpe - INFO - Computing initial pair counts from 576556 unique sequences
2026-05-27 15:27:38,799 - rustbpe - INFO - Building heap with 7119 unique pairs
2026-05-27 15:27:38,800 - rustbpe - INFO - Starting merge loop
2026-05-27 15:27:39,821 - rustbpe - INFO - Progress: 1% (326/32503 merges) - Last merge: (260, 114) -> 581 (frequency: 56548)
2026-05-27 15:27:40,046 - rustbpe - INFO - Progress: 2% (651/32503 merges) - Last merge: (65, 67) -> 906 (frequency: 22262)
2026-05-27 15:27:40,188 - rustbpe - INFO - Progress: 3% (976/32503 merges) - Last merge: (660, 742) -> 1231 (frequency: 134

In [ ]:
# ── Pretrain d12 base model (~2h on A100) ─────────────────────────────────────
# Checkpoints save every 500 steps to LOCAL/base_checkpoints/d12/ (-> Drive).
# If session dies, find latest checkpoint and re-run with --resume-from-step=N:
#
#   import glob, os
#   ckpts = sorted(glob.glob(f"{os.environ['DRIVE_ROOT']}/models/{os.environ['MODEL_NAME']}/base_checkpoints/d12/*"))
#   print([os.path.basename(c) for c in ckpts[-3:]])
%cd /content/nanochat
MODEL_DEPTH = int(os.environ["MODEL_DEPTH"])
!OMP_NUM_THREADS=1 python -m scripts.base_train \
    --depth={MODEL_DEPTH} \
    --window-pattern=L \
    --device-batch-size=16 \
    --save-every=500 \
    --core-metric-every=-1 \
    --sample-every=-1 \
    --run=dummy


/content/drive/MyDrive/Think.Genesis/nanochat

                                                       █████                █████
                                                      ░░███                ░░███
     ████████    ██████   ████████    ██████   ██████  ░███████    ██████  ███████
    ░░███░░███  ░░░░░███ ░░███░░███  ███░░███ ███░░███ ░███░░███  ░░░░░███░░░███░
     ░███ ░███   ███████  ░███ ░███ ░███ ░███░███ ░░░  ░███ ░███   ███████  ░███
     ░███ ░███  ███░░███  ░███ ░███ ░███ ░███░███  ███ ░███ ░███  ███░░███  ░███ ███
     ████ █████░░████████ ████ █████░░██████ ░░██████  ████ █████░░███████  ░░█████
    ░░░░ ░░░░░  ░░░░░░░░ ░░░░ ░░░░░  ░░░░░░   ░░░░░░  ░░░░ ░░░░░  ░░░░░░░░   ░░░░░
    
Autodetected device type: cuda
2026-05-27 15:51:46,467 - nanochat.common - INFO - Distributed world size: 1
GPU: NVIDIA A100-SXM4-40GB | Peak FLOPS (BF16): 3.12e+14
COMPUTE_DTYPE: torch.bfloat16 (auto-detected: CUDA SM 80 (bf16 supported))
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
# ── Sanity-check base model (text completions) ────────────────────────────────
# Run after Cell 6. If output is gibberish, base training failed — do not SFT.
%cd /content/nanochat
MODEL_TAG = os.environ["MODEL_TAG"]
!python -m scripts.base_eval --eval sample --model-tag {MODEL_TAG} --device-batch-size=16


In [ ]:
# ── Download SFT identity-conversation data ───────────────────────────────────
# SmolTalk, MMLU, GSM8K auto-download from HF on first SFT run.
# Only identity_conversations.jsonl needs a manual fetch.
import os
LOCAL = os.environ["LOCAL"]
!curl -L -o {LOCAL}/identity_conversations.jsonl \
    https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl
!ls -lh {LOCAL}/identity_conversations.jsonl


In [ ]:
# ── Supervised fine-tuning (~30-60 min) ──────────────────────────────────────
# Fine-tunes the base model. Checkpoints -> Drive via symlink.
%cd /content/nanochat
MODEL_TAG = os.environ["MODEL_TAG"]
!OMP_NUM_THREADS=1 python -m scripts.chat_sft \
    --model-tag={MODEL_TAG} \
    --device-batch-size=8 \
    --eval-every=-1 \
    --chatcore-every=-1 \
    --run=dummy


In [ ]:
# ── Launch chat web UI ────────────────────────────────────────────────────────
# Starts chat_web on port 8000 and opens a Colab proxy tunnel. Re-run safe.
import subprocess, time, os, urllib.request

MODEL_TAG = os.environ["MODEL_TAG"]

subprocess.run(["pkill", "-f", "scripts.chat_web"], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ["python", "-m", "scripts.chat_web",
     "-i", "sft", "-g", MODEL_TAG,
     "--port", "8000", "--host", "127.0.0.1"],
    cwd="/content/nanochat",
    env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(f"Server PID: {proc.pid}")
print("Waiting for boot (~30-90s)...")

for i in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print(f"Server up after ~{i*2}s")
        break
    except Exception:
        time.sleep(2)
        if proc.poll() is not None:
            print("Server process died:")
            print(proc.stdout.read() if proc.stdout else "(no output)")
            raise SystemExit
else:
    print("Server did not come up in 6 min. Recent logs:")
    print(proc.stdout.read() if proc.stdout else "(no output)")

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8000)")
print(f"Open this URL in a new tab:\n{url}")


In [ ]:
# ── Stop web server ───────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pkill", "-f", "scripts.chat_web"], capture_output=True)
print("Server stopped.")
